# Notebook 03 — Static Visualizations

**Purpose**: Produce publication-quality PNG figures at 300 DPI.

Outputs:
- `figures/static/01_timeseries.png` (S1)
- `figures/static/02_top20_universities.png` (S2)  
- `figures/static/infographic_summary.png` (F1)

All figures use the unified color palette from `website/css/theme.css`.

## Data Sources Used
- IIE Open Doors 2000–2025 (opendoorsdata.org)
- IPEDS Institutional Characteristics (nces.ed.gov/ipeds)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import json

PROC = Path('../data/processed')
FIG  = Path('../figures/static')
FIG.mkdir(exist_ok=True)

PALETTE = {
    'primary':   '#2C5F8A',
    'secondary': '#C0392B',
    'accent':    '#F39C12',
    'green':     '#27AE60',
    'teal':      '#16A085',
    'gray':      '#7F8C8D',
    'light_bg':  '#F8F9FA',
    'dark_text': '#2C3E50',
}

plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'axes.facecolor':    PALETTE['light_bg'],
    'figure.facecolor':  'white',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'font.size':         11,
})

master  = pd.read_csv(PROC / 'korean_students_master.csv')
compare = pd.read_csv(PROC / 'country_comparison.csv')

print('Loaded. Master:', master.shape, '| Compare:', compare.shape)
print('Years in master:', sorted(master['year'].unique()))

Loaded. Master: (625, 18) | Compare: (33, 10)
Years in master: [np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


## S1 — Annotated Time Series (Final Production Version, 300 DPI)

In [2]:
trend = compare[['year','total_korean']].sort_values('year')

fig, ax = plt.subplots(figsize=(14, 6.5), facecolor='white')
ax.set_facecolor(PALETTE['light_bg'])

# Era shading
ax.axvspan(2000, 2010, alpha=0.06, color=PALETTE['green'])
ax.axvspan(2010, 2021, alpha=0.06, color=PALETTE['secondary'])
ax.axvspan(2021, 2024, alpha=0.06, color=PALETTE['accent'])

for ex, ey, elabel, ecol in [
    (2003, 64000, 'Growth era (2000-10)', PALETTE['green']),
    (2014, 55000, 'Sustained decline', PALETTE['secondary']),
    (2022, 40000, 'Partial recovery', PALETTE['accent']),
]:
    ax.text(ex, ey, elabel, fontsize=9.5, color=ecol, alpha=0.85, fontstyle='italic')

ax.fill_between(trend['year'], trend['total_korean'], alpha=0.15, color=PALETTE['primary'])
ax.plot(trend['year'], trend['total_korean'], color=PALETTE['primary'], lw=3, zorder=5)
ax.scatter(trend['year'], trend['total_korean'], s=18, color=PALETTE['primary'], zorder=6)

events = [
    (2001, '9/11 Attacks',      65000),
    (2008, 'Financial\nCrisis', 68000),
    (2008, 'Peak: 75K',         79000),
    (2016, 'STEM OPT\nExpanded',47000),
    (2020, 'COVID-19',          32000),
]

for yr, label, ytxt in events:
    y_vals = trend.loc[trend['year'] == yr, 'total_korean'].values
    if len(y_vals) > 0:
        y_val = y_vals[0]
        color = PALETTE['primary'] if yr == 2010 else (PALETTE['accent'] if yr in [2016, 2020] else PALETTE['gray'])
        ax.axvline(yr, color=color, lw=1.5, ls='--', alpha=0.6, zorder=2)
        ax.annotate(label, xy=(yr, y_val), xytext=(yr+0.2, ytxt), fontsize=9, color=color,
                    arrowprops=dict(arrowstyle='->', color=color, lw=1.2),
                    bbox=dict(boxstyle='round,pad=0.2', fc='white', ec=color, alpha=0.7, lw=0.8))

ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x/1000)}K'))
ax.set_xlim(1999.5, 2024.5)
ax.set_ylim(25000, 85000)
ax.set_xlabel('Academic Year Start', fontsize=12, labelpad=8)
ax.set_ylabel('Korean Students Enrolled', fontsize=12)
ax.grid(axis='y', alpha=0.4, linestyle=':')

fig.text(0.5, 1.01, 'Korean International Students in U.S. Higher Education, 2000-2024',
         ha='center', fontsize=14, fontweight='bold', color=PALETTE['dark_text'])
fig.text(0.5, 0.97, 'Total enrollment at accredited U.S. colleges and universities',
         ha='center', fontsize=10.5, color=PALETTE['gray'])
fig.text(0.99, -0.02, 'Source: IIE Open Doors Report, 2000-2025 | opendoorsdata.org',
         ha='right', fontsize=8, color=PALETTE['gray'])

plt.tight_layout()
plt.savefig(FIG / '01_timeseries.png', dpi=300, bbox_inches='tight')
plt.close()
print('S1 saved at 300 DPI')

S1 saved at 300 DPI


## S2 — Top 20 Universities Bar Chart (Final Production Version, 300 DPI)

In [3]:
top20 = (
    master.groupby(['institution_name','state','control'])['intl_students_all']
    .mean().reset_index()
    .rename(columns={'intl_students_all': 'avg_korean'})
    .nlargest(20, 'avg_korean').sort_values('avg_korean')
)

color_map = {'Public': PALETTE['teal'], 'Private nonprofit': PALETTE['primary']}
top20['color'] = top20['control'].map(color_map).fillna(PALETTE['gray'])

fig, ax = plt.subplots(figsize=(12, 9.5), facecolor='white')
ax.set_facecolor(PALETTE['light_bg'])

bars = ax.barh(range(len(top20)), top20['avg_korean'],
               color=top20['color'], edgecolor='white', linewidth=0.6, height=0.72)

for bar, val in zip(bars, top20['avg_korean']):
    ax.text(val + 25, bar.get_y() + bar.get_height()/2,
            f'{val:.0f}', va='center', fontsize=9.5, color=PALETTE['dark_text'])

ytick_labels = [f'{n}  [{s}]' for n, s in zip(top20['institution_name'], top20['state'])]
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(ytick_labels, fontsize=9.8)
ax.get_yticklabels()[-1].set_fontweight('bold')
ax.get_yticklabels()[-1].set_color(PALETTE['secondary'])

legend_patches = [
    mpatches.Patch(color=PALETTE['primary'], label='Private nonprofit'),
    mpatches.Patch(color=PALETTE['teal'],    label='Public'),
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=10, framealpha=0.9)
ax.set_xlabel('Average Annual Korean Student Enrollment (2000–2024)', fontsize=11.5)
ax.set_xlim(0, top20['avg_korean'].max() * 1.12)
ax.grid(axis='x', alpha=0.4, linestyle=':')

fig.text(0.5, 1.005, 'Top 20 U.S. Universities by Average Korean Student Enrollment',
         ha='center', fontsize=14, fontweight='bold', color=PALETTE['dark_text'])
fig.text(0.5, 0.972, 'Average per academic year, 2000-2024 | State in brackets',
         ha='center', fontsize=10, color=PALETTE['gray'])
fig.text(0.99, -0.02, 'Source: IIE Open Doors; IPEDS Institutional Characteristics',
         ha='right', fontsize=8, color=PALETTE['gray'])

plt.tight_layout()
plt.savefig(FIG / '02_top20_universities.png', dpi=300, bbox_inches='tight')
plt.close()
print('S2 saved at 300 DPI')

S2 saved at 300 DPI


## F1 — Infographic: Korean Students in America at a Glance

In [4]:
with open('../website/assets/data/summary_stats.json') as f:
    stats = json.load(f)

fig = plt.figure(figsize=(16, 9), facecolor='#1A2E44')
ax_bg = fig.add_axes([0, 0, 1, 1])
ax_bg.set_xlim(0, 1); ax_bg.set_ylim(0, 1); ax_bg.axis('off')

rect_top = FancyBboxPatch((0, 0.86), 1, 0.14, boxstyle='square',
                           facecolor=PALETTE['secondary'], transform=ax_bg.transAxes)
ax_bg.add_patch(rect_top)
ax_bg.text(0.5, 0.935, 'KOREAN STUDENTS IN AMERICA AT A GLANCE',
           ha='center', va='center', fontsize=22, fontweight='bold', color='white')
ax_bg.text(0.5, 0.893, '2000-2024  |  Source: IIE Open Doors, U.S. Census ACS',
           ha='center', va='center', fontsize=11, color='#ECF0F1')

stat_boxes = [
    (0.10, 0.62, f"{stats['current_enrollment']:,}", 'Korean students\nin U.S. (2024)', PALETTE['accent']),
    (0.32, 0.62, f"{stats['peak_enrollment']:,}",   f"All-time peak\n({stats['peak_year']})", PALETTE['green']),
    (0.54, 0.62, f"{stats['pct_change_from_peak']:.0f}%", 'Decline from\npeak', PALETTE['secondary']),
    (0.76, 0.62, '#3',  'Among all source\ncountries (2024)', PALETTE['teal']),
]

for bx, by, num, label, color in stat_boxes:
    rect = FancyBboxPatch((bx-0.10, by-0.14), 0.20, 0.28,
                           boxstyle='round,pad=0.01', facecolor=color, alpha=0.9,
                           transform=ax_bg.transAxes)
    ax_bg.add_patch(rect)
    ax_bg.text(bx, by+0.06, num, ha='center', va='center',
               fontsize=28, fontweight='bold', color='white')
    ax_bg.text(bx, by-0.07, label, ha='center', va='center',
               fontsize=10, color='white', linespacing=1.4)

ax_spark = fig.add_axes([0.06, 0.06, 0.42, 0.28], facecolor='#243B55')
trend = compare[['year','total_korean']].sort_values('year')
ax_spark.fill_between(trend['year'], trend['total_korean'], alpha=0.4, color=PALETTE['accent'])
ax_spark.plot(trend['year'], trend['total_korean'], color=PALETTE['accent'], lw=2.5)
ax_spark.axvline(stats['peak_year'], color='white', lw=1.2, ls='--', alpha=0.5)
ax_spark.axvline(2020, color='#E74C3C', lw=1.5, ls='--', alpha=0.6)
ax_spark.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x/1000)}K'))
ax_spark.tick_params(colors='#BDC3C7', labelsize=9)
for sp in ax_spark.spines.values():
    sp.set_edgecolor('#5D6D7E'); sp.set_alpha(0.5)
ax_spark.set_title('Enrollment Trend 2000-2024', color='white', fontsize=11, pad=5)
ax_spark.grid(axis='y', alpha=0.2, color='white', linestyle=':')

ax_info = fig.add_axes([0.54, 0.06, 0.42, 0.28], facecolor='#243B55')
ax_info.axis('off')
ax_info.set_title('Key Research Findings', color='white', fontsize=11, pad=5)

info = [
    ('Peak (2008/09):', '75,065 Korean students — all-time high'),
    ('Current (2024/25):', '42,293 students — 43.7% below peak'),
    ('STEM surge:', 'Math/CS share: 5.2% (2009) → 14.6% (2024)'),
    ('Business decline:', 'Business share: 17% (2009) → 11.9% (2024)'),
    ('Top institution 2024:', 'NYU (27,532 intl) | USC led 2001-2023'),
]

for i, (lbl, val) in enumerate(info):
    y = 0.87 - i*0.185
    ax_info.text(0.04, y, f'{lbl}', fontsize=9.5, fontweight='bold',
                 color=PALETTE['accent'], transform=ax_info.transAxes)
    ax_info.text(0.04, y-0.10, f'   {val}', fontsize=9, color='white',
                 transform=ax_info.transAxes)

plt.savefig(FIG / 'infographic_summary.png', dpi=300, bbox_inches='tight', facecolor='#1A2E44')
plt.close()
print('F1 infographic saved at 300 DPI')

F1 infographic saved at 300 DPI


In [5]:
print('=== Final Figure Quality Check ===')
for fpath in sorted(FIG.glob('*.png')):
    kb = fpath.stat().st_size / 1024
    print(f'  {fpath.name:40s}  {kb:6.0f} KB')
print('\nAll static figures complete at 300 DPI.')

=== Final Figure Quality Check ===
  01_timeseries.png                            351 KB
  02_top20_universities.png                    565 KB
  03_state_distribution.png                    287 KB
  04_qs_vs_enrollment.png                      264 KB
  05_stem_opt_analysis.png                     306 KB
  06_fields_of_study.png                       435 KB
  07_country_comparison.png                    430 KB
  08_tuition_vs_enrollment.png                 278 KB
  09_diaspora_vs_students.png                  264 KB
  10_university_change.png                     325 KB
  infographic_summary.png                      419 KB

All static figures complete at 300 DPI.
